In [ ]:
import re, json
import numpy as np
import pandas as pd

TRAIN_CSV   = "train.csv"
PROMPT_FILE = "prompt_attack_v2.txt"
SEED        = 2026

# field order + types from section 4 of the prompt
_txt = open(PROMPT_FILE).read()
_fields = re.findall(r"^\d+\. (\w+) \((whole number|decimal|text)\)", _txt, flags=re.M)
FIELD_ORDER = [f for f, _ in _fields]
WHOLE = {f for f, t in _fields if t == "whole number"}
assert len(FIELD_ORDER) == 113

train = pd.read_csv(TRAIN_CSV)
atk = train[train["is_attack"] == True].reset_index(drop=True)   # TRAIN attack rows only
rng = np.random.default_rng(SEED)

is_long = atk["pkts_fwd"] > 1
full    = atk["timeframe_id"].between(5, 10)
wind    = atk["timeframe_id"] == 11
ports   = list(rng.permutation(sorted(atk["port_b"].unique())))   # 8 ports, shuffled

picked = []
def take(mask):
    idx = atk.index[mask & ~atk.index.isin(picked)]
    picked.append(rng.choice(idx))

# 8 long flows covering all 8 ports: one per full-attack timeframe (5-10),
# one extra full-attack flow, one wind-down flow (tf 11)
for tf, p in zip([5, 6, 7, 8, 9, 10], ports[:6]):
    take(is_long & (atk["timeframe_id"] == tf) & (atk["port_b"] == p))
take(is_long & full & (atk["port_b"] == ports[6]))
take(is_long & wind & (atk["port_b"] == ports[7]))
# 2 single-packet flows (~16% of real rows): one full-attack, one wind-down
take(~is_long & full)
take(~is_long & wind)
rng.shuffle(picked)

def to_record(row):
    rec = {}
    for f in FIELD_ORDER:
        v = row[f]
        if f == "is_attack":                      rec[f] = "True"
        elif pd.isna(v):                          rec[f] = None
        elif f in ("ip_a", "ip_b", "flow_key"):   rec[f] = str(v)
        elif f in WHOLE:                          rec[f] = int(round(float(v)))
        else:                                     rec[f] = float(v)
    return rec

examples = [to_record(atk.loc[i]) for i in picked]
print(atk.loc[picked, ["timeframe_id", "port_b", "pkts_fwd", "duration"]], "\n")

atk.loc[picked].to_csv("prompt_examples.csv", index=False)      # for the copy check later
block = "\n".join(json.dumps(r, separators=(",", ":")) for r in examples)
open("examples_block.txt", "w").write(block)
print(f"{len(block)} chars ~ {len(block)//3} tokens -> saved examples_block.txt")